In [2]:
import os
import re

repo_name = 'OmniFile_Processor'

# Check if the directory exists and navigate into it, otherwise clone and then navigate
if os.path.exists(repo_name):
    print(f"Directory '{repo_name}' already exists. Skipping clone.")
    %cd {repo_name}
else:
    print(f"Cloning '{repo_name}'...")
    !git clone https://github.com/DrAbdulmalek/OmniFile_Processor.git
    %cd {repo_name}

!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng libgl1
!pip install -r requirements-colab.txt

# Patch hf_app.py to fix Gradio compatibility error and enable public link
if os.path.exists('hf_app.py'):
    with open('hf_app.py', 'r') as f:
        content = f.read()

    # Fix Gradio compatibility error: replace multiple=True with file_count='multiple'
    patched_content = content.replace("multiple=True", "file_count='multiple'")

    # More robust logic for 'share=True'
    # Find the last occurrence of '.launch('
    last_launch_start = patched_content.rfind('.launch(')
    if last_launch_start != -1:
        # Find the corresponding closing parenthesis, accounting for nesting
        nesting_level = 0
        launch_args_start = last_launch_start + len('.launch(')
        launch_args_end = -1

        for i in range(launch_args_start, len(patched_content)):
            if patched_content[i] == '(': # Increase nesting level for open parenthesis
                nesting_level += 1
            elif patched_content[i] == ')': # Decrease nesting level for close parenthesis
                if nesting_level == 0: # If nesting level is 0, this is the matching closing parenthesis
                    launch_args_end = i
                    break
                nesting_level -= 1

        if launch_args_end != -1:
            # Extract the arguments string
            args_str = patched_content[launch_args_start:launch_args_end]

            new_args_str = args_str # Initialize new_args_str with original

            # Check if 'share=False' is present and replace it
            if 'share=False' in args_str:
                new_args_str = args_str.replace('share=False', 'share=True')
            elif 'share=True' not in args_str: # If share=False not found and share=True not found
                # Append 'share=True', ensuring correct comma separation
                if args_str.strip(): # if there are existing arguments
                    # Only add a comma if the last non-whitespace char is not already a comma
                    if not args_str.strip().endswith(','):
                        new_args_str = args_str.rstrip() + ', share=True'
                    else: # if it already ends with a comma, just add a space before 'share=True'
                        new_args_str = args_str.rstrip() + ' share=True'
                else: # no arguments at all
                    new_args_str = 'share=True'
            # else: share=True already exists, no change needed (new_args_str remains args_str)

            # Reconstruct the patched content with the modified arguments string
            patched_content = (
                patched_content[:launch_args_start] +
                new_args_str +
                patched_content[launch_args_end:]
            )

    with open('hf_app.py', 'w') as f:
        f.write(patched_content)

!python hf_app.py

Directory 'OmniFile_Processor' already exists. Skipping clone.
/content/OmniFile_Processor/OmniFile_Processor
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
22:40:05 [INFO] OmniFile_HF: EngineRouter loaded successfully
22:40:06 [INFO] OmniFile_HF: No GPU detected — running on CPU
22:40:06 [INFO] OmniFile_HF: ============================================================
22:40:06 [INFO] OmniFile_HF:   OmniFile AI Processor — HF Spaces
22:40:06 [INFO] OmniFile_HF:   Device : cpu  |  GPU: N/A
22:40:06 [INFO] OmniFile_HF:   Cache  : /data/.cache/huggingface
22:40:06 [INFO] OmniFile_HF: ============================================================
22:40:06 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
22:40:07 [WARNING] OmniFile_HF: Word Trainer tabs unavailable:

In [3]:
import os, glob, zipfile
from google.colab import files

# Find the output folder for this run
folders = glob.glob('/tmp/omnifile_train_*/output')
if not folders:
    print("No training output folder found. Has the runtime restarted?")
else:
    output_dir = folders[0]
    zip_path = '/content/omnifile_training_data.zip'

    with zipfile.ZipFile(zip_path, 'w') as zf:
        # Add the training output (train.jsonl, val.jsonl, images, etc.)
        for root, dirs, filenames in os.walk(output_dir):
            for fname in filenames:
                full_path = os.path.join(root, fname)
                arcname = os.path.relpath(full_path, output_dir)
                zf.write(full_path, arcname)

        # Add the exported corrections
        corrections_file = '/tmp/omnifile_corrections_export.json'
        if os.path.exists(corrections_file):
            zf.write(corrections_file, os.path.basename(corrections_file))

    print("Downloading...")
    files.download(zip_path)

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import json
import glob
import os
import gradio as gr
from PIL import Image

# ── 1. البحث عن ملفات JSONL والصور ──────────────────────
output_dirs = glob.glob('/tmp/omnifile_train_*/output/input')
if not output_dirs:
    raise FileNotFoundError("لم يتم العثور على مجلد الإخراج. ربما تم إعادة تشغيل Colab؟")

base_dir = output_dirs[0]  # مجلد input/
jsonl_files = [f for f in os.listdir(base_dir) if f.endswith('.jsonl')]

all_samples = []
for jf in jsonl_files:
    with open(os.path.join(base_dir, jf), 'r', encoding='utf-8') as f:
        for line in f:
            sample = json.loads(line)
            # المسار النسبي للصورة (عادة "images/word_001.png")
            img_path = os.path.join(base_dir, sample.get('file', sample.get('image', '')))
            all_samples.append({
                'image': img_path,
                'text': sample.get('text', ''),
                'source_file': jf
            })

# ── 2. دوال الواجهة ───────────────────────────────────
current_idx = 0
total = len(all_samples)

def load_sample(idx):
    """تحميل العينة رقم idx وعرض الصورة والنص"""
    if 0 <= idx < total:
        s = all_samples[idx]
        return s['image'], s['text'], f"العينة {idx+1} / {total}"
    return None, "", "لا توجد بيانات"

def save_and_next(current_text, idx):
    """حفظ النص المعدل والانتقال للتالي"""
    if 0 <= idx < total:
        all_samples[idx]['text'] = current_text
    next_idx = idx + 1
    if next_idx >= total:
        return [None, "", "تم الانتهاء من جميع العينات! اضغط 'حفظ النتائج' لتحميل الملفات."]
    return load_sample(next_idx)

def save_and_prev(current_text, idx):
    """حفظ النص المعدل والرجوع للسابق"""
    if 0 <= idx < total:
        all_samples[idx]['text'] = current_text
    prev_idx = idx - 1
    if prev_idx < 0:
        prev_idx = 0
    return load_sample(prev_idx)

def export_corrected():
    """تصدير جميع التصحيحات كملفات JSONL جديدة"""
    export_dir = '/content/corrected_dataset'
    os.makedirs(export_dir, exist_ok=True)
    # تجميع حسب الملف المصدر
    by_source = {}
    for s in all_samples:
        by_source.setdefault(s['source_file'], []).append(s)
    exported_files = []
    for src, items in by_source.items():
        out_path = os.path.join(export_dir, f"corrected_{src}")
        with open(out_path, 'w', encoding='utf-8') as f:
            for item in items:
                f.write(json.dumps({"file": os.path.basename(item['image']),
                                    "text": item['text']}, ensure_ascii=False) + '\n')
        exported_files.append(out_path)
    # إنشاء ZIP
    import zipfile
    zip_path = '/content/corrected_dataset.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for ef in exported_files:
            zf.write(ef, os.path.basename(ef))
    return zip_path, "تم حفظ الملفات المصححة. اضغط على الرابط للتحميل."

# ── 3. بناء واجهة Gradio ─────────────────────────────
with gr.Blocks(title="مراجعة النصوص اليدوية") as demo:
    gr.Markdown("## مراجعة الكلمات المستخرجة وتصحيح النصوص")

    with gr.Row():
        with gr.Column(scale=2):
            image = gr.Image(type="filepath", label="صورة الكلمة")
        with gr.Column(scale=3):
            text_edit = gr.Textbox(label="النص المتوقع (يمكنك تعديله)", lines=2)
            info = gr.Label(value="")
            with gr.Row():
                prev_btn = gr.Button("السابق")
                next_btn = gr.Button("التالي")
            export_btn = gr.Button("حفظ النتائج النهائية")
            download_file = gr.File(label="النتائج المصححة")

    # الحالة المخفية
    idx_state = gr.State(0)

    # تحميل العينة الأولى
    demo.load(lambda: load_sample(0), outputs=[image, text_edit, info])

    # التفاعل
    next_btn.click(
        save_and_next,
        inputs=[text_edit, idx_state],
        outputs=[image, text_edit, info]
    ).then(
        lambda idx: idx + 1,
        inputs=[idx_state],
        outputs=[idx_state]
    )

    prev_btn.click(
        save_and_prev,
        inputs=[text_edit, idx_state],
        outputs=[image, text_edit, info]
    ).then(
        lambda idx: max(0, idx - 1),
        inputs=[idx_state],
        outputs=[idx_state]
    )

    export_btn.click(
        export_corrected,
        outputs=[download_file, info]
    )

demo.launch(share=True, server_port=7861)  # منفذ مختلف تجنبًا للتعارض

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1297a3531488b85094.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
